In [1]:
import os
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")
!mkdir -p ~/.kaggle
!echo "{token}" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
!git clone https://{GITHUB_TOKEN}@github.com/amama22/walmart-forecasting
%cd walmart-forecasting

!kaggle competitions download \
  -c walmart-recruiting-store-sales-forecasting \
  -p data

import zipfile
data_dir = "data"
while any(f.endswith(".zip") for f in os.listdir(data_dir)):
    for fname in list(os.listdir(data_dir)):
        if fname.endswith(".zip"):
            fpath = os.path.join(data_dir, fname)
            with zipfile.ZipFile(fpath, "r") as z:
                z.extractall(data_dir)
            os.remove(fpath)

%pip install -q dagshub
!pip install mlflow --quiet

import dagshub
import mlflow

dagshub.init(repo_owner="amama22", repo_name="walmart-forecasting", mlflow=True)
mlflow.set_experiment("NBEATS_Training")

Cloning into 'walmart-forecasting'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 36 (delta 13), reused 19 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 215.58 KiB | 5.67 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/walmart-forecasting
100% 2.70M/2.70M [00:00<00:00, 52.0MB/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=41bea856-f60c-48ac-8d30-fe7e54d985f7&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=6f984b9b9a30ad89d518de1635f5c1e715eaca3b8a8d83ab86219391ef099d15




Accessing as amama22

Initialized MLflow to track repo "amama22/walmart-forecasting"

Repository amama22/walmart-forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/c57d5a176c2e4b8aa10b3c90deaccd59', creation_time=1783801585600, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783801585600, lifecycle_stage='active', name='NBEATS_Training', tags={}, trace_location=None, workspace='default'>

In [2]:
%pip install -q "u8darts[torch]"

import numpy, torch, pytorch_lightning, darts
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("pytorch_lightning:", pytorch_lightning.__version__)
print("darts:", darts.__version__)
print("CUDA available:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.5/744.5 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.6/413.6 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 53.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour i

In [4]:
import pandas as pd
import numpy as np
import sys
sys.path.append(".")

from darts import TimeSeries
from darts.models import NBEATSModel

from src.data_prep import load_raw_data, merge_all
from src.evaluation import wmae

train, test, features, stores = load_raw_data(data_dir="data")
train_merged = merge_all(train, features, stores)

df = train_merged[["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]].copy()
df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
df = df.sort_values(["unique_id", "Date"])

# small subset first -- 20 series, few epochs, just to prove fit->predict runs clean
subset_ids = df["unique_id"].unique()[:20]
df_subset = df[df["unique_id"].isin(subset_ids)]

series_list = TimeSeries.from_group_dataframe(
    df_subset, group_cols="unique_id", time_col="Date", value_cols="Weekly_Sales", freq="W-FRI", fill_missing_dates=True, fillna_value=0,
)
print("Number of series built:", len(series_list))
print("First series length:", len(series_list[0]))

train_series = [s[:-38] for s in series_list]
val_series = [s[-38:] for s in series_list]

smoke_model = NBEATSModel(
    input_chunk_length=104,
    output_chunk_length=38,
    n_epochs=3,
    random_state=42,
    pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
)

smoke_model.fit(series_list)
preds = smoke_model.predict(n=38, series=series_list)

print("Smoke test completed. Number of predictions:", len(preds))
print("First prediction values (head):", preds[0].values()[:5].flatten())

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Number of series built: 20
First series length: 143


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 6.8 M  | train
-------------------------------------------------------------
6.8 M     Trainable params
1.9 K     Non-trainable params
6.8 M     Total params
27.321    Total estimated model params size (MB)
396       Modules in train mode
0         Modules in eval mode
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, Tre

Smoke test completed. Number of predictions: 20
First prediction values (head): [46279.3054774  31169.66357858 24105.73462384 38746.7546482
 46057.30741665]


In [5]:
subset_ids_500 = df["unique_id"].unique()[:500]
df_subset_500 = df[df["unique_id"].isin(subset_ids_500)]

series_list_500 = TimeSeries.from_group_dataframe(
    df_subset_500, group_cols="unique_id", time_col="Date", value_cols="Weekly_Sales", freq="W-FRI", fill_missing_dates=True, fillna_value=0,
)
print("Number of series built:", len(series_list_500))

smoke_model_500 = NBEATSModel(
    input_chunk_length=104,
    output_chunk_length=38,
    n_epochs=3,
    random_state=42,
    pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
)

smoke_model_500.fit(series_list_500)
preds_500 = smoke_model_500.predict(n=38, series=series_list_500)

print("500-series smoke test completed. Number of predictions:", len(preds_500))

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 6.8 M  | train
-------------------------------------------------------------
6.8 M     Trainable params
1.9 K     Non-trainable params
6.8 M     Total params
27.321    Total estimated m

Number of series built: 500


ValueError: The dataset contains target `series` that are too short to extract even a single example. Expected min length: `142`, received length `138` (at series sequence idx `38`).

In [6]:
from darts import TimeSeries

MIN_LEN = 104 + 38  # input_chunk_length + output_chunk_length

def pad_if_short(series, min_len=MIN_LEN):
    if len(series) < min_len:
        pad_len = min_len - len(series)
        series = series.prepend_values(np.zeros(pad_len))
    return series

series_list_500 = [pad_if_short(s) for s in series_list_500]

lengths_after_pad = [len(s) for s in series_list_500]
print("Min length after padding:", min(lengths_after_pad))
print("Number of series padded:", sum(1 for s in series_list_500 if len(s) == MIN_LEN))

smoke_model_500 = NBEATSModel(
    input_chunk_length=104,
    output_chunk_length=38,
    n_epochs=3,
    random_state=42,
    pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
)

smoke_model_500.fit(series_list_500)
preds_500 = smoke_model_500.predict(n=38, series=series_list_500)

print("500-series smoke test completed. Number of predictions:", len(preds_500))

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 6.8 M  | train
-------------------------------------------------------------
6.8 M     Trainable params
1.9 K     Non-trainable params
6.8 M     Total params
27.321    Total estimated m

Min length after padding: 142
Number of series padded: 48


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


500-series smoke test completed. Number of predictions: 500


In [7]:
subset_ids_full = df["unique_id"].unique()
df_full = df[df["unique_id"].isin(subset_ids_full)]

series_list_full = TimeSeries.from_group_dataframe(
    df_full, group_cols="unique_id", time_col="Date", value_cols="Weekly_Sales", freq="W-FRI", fill_missing_dates=True, fillna_value=0,
)

series_list_full = [pad_if_short(s) for s in series_list_full]

print("Total series:", len(series_list_full))
print("Min length after padding:", min(len(s) for s in series_list_full))
print("Series padded:", sum(1 for s in series_list_full if len(s) == MIN_LEN))

smoke_model_full = NBEATSModel(
    input_chunk_length=104,
    output_chunk_length=38,
    n_epochs=3,
    random_state=42,
    pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
)

smoke_model_full.fit(series_list_full)
preds_full = smoke_model_full.predict(n=38, series=series_list_full)

print("Full-scale smoke test completed. Number of predictions:", len(preds_full))

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 6.8 M  | train
-------------------------------------------------------------
6.8 M     Trainable params
1.9 K     Non-trainable params
6.8 M     Total params
27.321    Total estimated m

Total series: 3331
Min length after padding: 142
Series padded: 478


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Full-scale smoke test completed. Number of predictions: 3331


In [10]:
series_list_raw = TimeSeries.from_group_dataframe(
    df_full, group_cols="unique_id", time_col="Date", value_cols="Weekly_Sales", freq="W-FRI", fill_missing_dates=True, fillna_value=0,
)

train_cutoff = fold_cutoffs[0][0]
print("Fit boundary (train cutoff):", train_cutoff)

train_series_list = [s.drop_after(pd.Timestamp(train_cutoff)) for s in series_list_raw]
train_series_list = [pad_if_short(s) for s in train_series_list]

print("Min train-series length after truncation+pad:", min(len(s) for s in train_series_list))

with mlflow.start_run(run_name="NBEATS_Baseline"):
    mlflow.log_param("input_chunk_length", 104)
    mlflow.log_param("output_chunk_length", HORIZON)
    mlflow.log_param("n_epochs", 30)
    mlflow.log_param("train_cutoff", str(train_cutoff.date()))

    model = NBEATSModel(
        input_chunk_length=104,
        output_chunk_length=HORIZON,
        n_epochs=30,
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
    )
    model.fit(train_series_list)

    unique_ids = df["unique_id"].unique()
    fold_wmaes = []
    for i, (val_start, val_end) in enumerate(fold_cutoffs):
        context_series = [pad_if_short(s.drop_after(pd.Timestamp(val_start))) for s in series_list_raw]
        preds = model.predict(n=HORIZON, series=context_series)

        y_true, y_pred, is_holiday = [], [], []
        for uid, raw_series, pred in zip(unique_ids, series_list_raw, preds):
            actual_slice = raw_series.slice(pd.Timestamp(val_start) + pd.Timedelta(days=7), pd.Timestamp(val_end))
            common_len = min(len(actual_slice), len(pred))
            y_true.extend(actual_slice.values().flatten()[:common_len])
            y_pred.extend(pred.values().flatten()[:common_len])

            holiday_vals = df[(df["unique_id"] == uid) & (df["Date"] > val_start) & (df["Date"] <= val_end)]["IsHoliday"].values
            is_holiday.extend(holiday_vals[:common_len])

        fold_wmae = wmae(y_true, y_pred, is_holiday)
        fold_wmaes.append(fold_wmae)
        mlflow.log_metric(f"fold{i}_wmae", fold_wmae)
        print(f"Fold {i}: WMAE = {fold_wmae:.2f}")

    mean_wmae = np.mean(fold_wmaes)
    mlflow.log_metric("mean_wmae", mean_wmae)
    print(f"\nMean WMAE (N-BEATS baseline): {mean_wmae:.2f}")

ERROR:darts.timeseries:ValueError: Timestamp must be between 2011-11-25 00:00:00 and 2011-11-25 00:00:00


Fit boundary (train cutoff): 2010-08-20 00:00:00


ValueError: Timestamp must be between 2011-11-25 00:00:00 and 2011-11-25 00:00:00

In [13]:
df_global = (
    df.groupby("unique_id", group_keys=True)
    .apply(reindex_global, include_groups=False)
    .reset_index()
)

print(df_global.columns.tolist())
print("Shape before:", df.shape, "-> after global reindex:", df_global.shape)
print("Rows added:", df_global.shape[0] - df.shape[0])

series_list_raw = TimeSeries.from_group_dataframe(
    df_global, group_cols="unique_id", time_col="Date", value_cols="Weekly_Sales", freq="W-FRI",
)

print("All series now same length:", len(set(len(s) for s in series_list_raw)) == 1, "-> length:", len(series_list_raw[0]))

['unique_id', 'Date', 'Store', 'Dept', 'Weekly_Sales', 'IsHoliday']
Shape before: (421570, 6) -> after global reindex: (476333, 6)
Rows added: 54763
All series now same length: True -> length: 143


In [14]:
last_date = df_global["Date"].max()
horizon_days = HORIZON * 7

fold_cutoffs = []
for i in range(3):
    val_end = last_date - pd.Timedelta(days=horizon_days * i)
    val_start = val_end - pd.Timedelta(days=horizon_days)
    fold_cutoffs.append((val_start, val_end))
fold_cutoffs = list(reversed(fold_cutoffs))

train_cutoff = fold_cutoffs[0][0]
print("Fit boundary (train cutoff):", train_cutoff)
for i, (vs, ve) in enumerate(fold_cutoffs):
    print(f"Fold {i}: {vs.date()} to {ve.date()}")

train_series_list = [s.drop_after(pd.Timestamp(train_cutoff)) for s in series_list_raw]
print("Min train-series length after truncation:", min(len(s) for s in train_series_list))

with mlflow.start_run(run_name="NBEATS_Baseline"):
    mlflow.log_param("input_chunk_length", 104)
    mlflow.log_param("output_chunk_length", HORIZON)
    mlflow.log_param("n_epochs", 30)
    mlflow.log_param("train_cutoff", str(train_cutoff.date()))

    model = NBEATSModel(
        input_chunk_length=104,
        output_chunk_length=HORIZON,
        n_epochs=30,
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
    )
    model.fit(train_series_list)

    unique_ids = df_global["unique_id"].unique()
    fold_wmaes = []
    for i, (val_start, val_end) in enumerate(fold_cutoffs):
        context_series = [s.drop_after(pd.Timestamp(val_start)) for s in series_list_raw]
        preds = model.predict(n=HORIZON, series=context_series)

        y_true, y_pred, is_holiday = [], [], []
        for uid, raw_series, pred in zip(unique_ids, series_list_raw, preds):
            actual_slice = raw_series.slice(pd.Timestamp(val_start) + pd.Timedelta(days=7), pd.Timestamp(val_end))
            common_len = min(len(actual_slice), len(pred))
            y_true.extend(actual_slice.values().flatten()[:common_len])
            y_pred.extend(pred.values().flatten()[:common_len])

            holiday_vals = df_global[(df_global["unique_id"] == uid) & (df_global["Date"] > val_start) & (df_global["Date"] <= val_end)]["IsHoliday"].values
            is_holiday.extend(holiday_vals[:common_len])

        fold_wmae = wmae(y_true, y_pred, is_holiday)
        fold_wmaes.append(fold_wmae)
        mlflow.log_metric(f"fold{i}_wmae", fold_wmae)
        print(f"Fold {i}: WMAE = {fold_wmae:.2f}")

    mean_wmae = np.mean(fold_wmaes)
    mlflow.log_metric("mean_wmae", mean_wmae)
    print(f"\nMean WMAE (N-BEATS baseline): {mean_wmae:.2f}")

Fit boundary (train cutoff): 2010-08-20 00:00:00
Fold 0: 2010-08-20 to 2011-05-13
Fold 1: 2011-05-13 to 2012-02-03
Fold 2: 2012-02-03 to 2012-10-26
Min train-series length after truncation: 28


ERROR:main_logger:ValueError: The input `series` are too short to extract even a single sample. Expected min length: `142`, received max length: `28`.


🏃 View run NBEATS_Baseline at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/642fd7988faa4d7c89a690e80f22a821
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


ValueError: The input `series` are too short to extract even a single sample. Expected min length: `142`, received max length: `28`.

In [15]:
val_end = last_date
val_start = val_end - pd.Timedelta(days=HORIZON * 7)
train_cutoff = val_start

print("Train cutoff:", train_cutoff.date())
print("Validation window:", val_start.date(), "to", val_end.date())

INPUT_LEN = 52

train_series_list = [s.drop_after(pd.Timestamp(train_cutoff)) for s in series_list_raw]
print("Min train-series length after truncation:", min(len(s) for s in train_series_list))

with mlflow.start_run(run_name="NBEATS_Baseline"):
    mlflow.log_param("input_chunk_length", INPUT_LEN)
    mlflow.log_param("output_chunk_length", HORIZON)
    mlflow.log_param("n_epochs", 30)
    mlflow.log_param("n_folds", 1)
    mlflow.log_param("train_cutoff", str(train_cutoff.date()))

    model = NBEATSModel(
        input_chunk_length=INPUT_LEN,
        output_chunk_length=HORIZON,
        n_epochs=30,
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
    )
    model.fit(train_series_list)

    context_series = [s.drop_after(pd.Timestamp(val_start)) for s in series_list_raw]
    preds = model.predict(n=HORIZON, series=context_series)

    unique_ids = df_global["unique_id"].unique()
    y_true, y_pred, is_holiday = [], [], []
    for uid, raw_series, pred in zip(unique_ids, series_list_raw, preds):
        actual_slice = raw_series.slice(pd.Timestamp(val_start) + pd.Timedelta(days=7), pd.Timestamp(val_end))
        common_len = min(len(actual_slice), len(pred))
        y_true.extend(actual_slice.values().flatten()[:common_len])
        y_pred.extend(pred.values().flatten()[:common_len])

        holiday_vals = df_global[(df_global["unique_id"] == uid) & (df_global["Date"] > val_start) & (df_global["Date"] <= val_end)]["IsHoliday"].values
        is_holiday.extend(holiday_vals[:common_len])

    fold_wmae = wmae(y_true, y_pred, is_holiday)
    mlflow.log_metric("fold0_wmae", fold_wmae)
    mlflow.log_metric("mean_wmae", fold_wmae)
    print(f"WMAE (single held-out fold): {fold_wmae:.2f}")

Train cutoff: 2012-02-03
Validation window: 2012-02-03 to 2012-10-26
Min train-series length after truncation: 104


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 6.4 M  | train
-------------------------------------------------------------
6.4 M     Trainable params
1.6 K     Non-trainable params
6.4 M     Total params
25.686    Total estimated m

WMAE (single held-out fold): 1793.25
🏃 View run NBEATS_Baseline at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/795337e54579490c84a247ca4034ec1a
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


In [ ]:
%pip install -q optuna
import optuna

def objective(trial):
    input_len = trial.suggest_int("input_chunk_length", 26, 66)
    num_stacks = trial.suggest_int("num_stacks", 2, 6)
    num_blocks = trial.suggest_int("num_blocks", 1, 3)
    layer_widths = trial.suggest_categorical("layer_widths", [128, 256, 512])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    model = NBEATSModel(
        input_chunk_length=input_len,
        output_chunk_length=HORIZON,
        num_stacks=num_stacks,
        num_blocks=num_blocks,
        layer_widths=layer_widths,
        optimizer_kwargs={"lr": lr},
        n_epochs=15,
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
    )
    model.fit(train_series_list)

    context_series = [s.drop_after(pd.Timestamp(val_start)) for s in series_list_raw]
    preds = model.predict(n=HORIZON, series=context_series)

    y_true, y_pred, is_holiday = [], [], []
    for uid, raw_series, pred in zip(unique_ids, series_list_raw, preds):
        actual_slice = raw_series.slice(pd.Timestamp(val_start) + pd.Timedelta(days=7), pd.Timestamp(val_end))
        common_len = min(len(actual_slice), len(pred))
        y_true.extend(actual_slice.values().flatten()[:common_len])
        y_pred.extend(pred.values().flatten()[:common_len])
        holiday_vals = df_global[(df_global["unique_id"] == uid) & (df_global["Date"] > val_start) & (df_global["Date"] <= val_end)]["IsHoliday"].values
        is_holiday.extend(holiday_vals[:common_len])

    trial_wmae = wmae(y_true, y_pred, is_holiday)

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        mlflow.log_params({"input_chunk_length": input_len, "num_stacks": num_stacks, "num_blocks": num_blocks, "layer_widths": layer_widths, "lr": lr, "n_epochs": 15})
        mlflow.log_metric("wmae", trial_wmae)

    return trial_wmae

with mlflow.start_run(run_name="NBEATS_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=30)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_wmae", study.best_value)
    print("Best WMAE:", study.best_value)
    print("Best params:", study.best_params)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.1 MB/s eta 0:00:00


[I 2026-07-11 22:49:15,064] A new study created in memory with name: no-name-9bbe9e48-8254-4b1f-adc8-3b7b37f73008
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 351 K  | train
--------------------------------------------------------

🏃 View run trial_0 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/f2232b21c979408dacb329834d6d4751
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 9.8 M  | train
-------------------------------------------------------------
9.8 M     Trainable params
2.8 K     Non-trainable params
9.8 M     Total params
39.248    Total estimated m

🏃 View run trial_1 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/3fd592e1954841a690840843c39d5b2a
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


...

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 228 K  | train
-------------------------------------------------------------
227 K     Trainable params
909       Non-trainable params
228 K     Total params
0.913     Total estimated m

🏃 View run trial_25 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/eade1717354b40919066284b264c0356
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 176 K  | train
-------------------------------------------------------------
175 K     Trainable params
987       Non-trainable params
176 K     Total params
0.706     Total estimated model params size (MB)
45        Modules in train mode
0         Modules in eval mode
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:

🏃 View run trial_26 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/9bd1ca84377c475f91bda0050a150d3e
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 691 K  | train
-------------------------------------------------------------
690 K     Trainable params
933       Non-trainable params
691 K     Total params
2.765     Total estimated model params size (MB)
146       Modules in train mode
0         Modules in eval mode
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


In [1]:
import os
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")
!mkdir -p ~/.kaggle
!echo "{token}" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
!git clone https://{GITHUB_TOKEN}@github.com/amama22/walmart-forecasting
%cd walmart-forecasting

!kaggle competitions download \
  -c walmart-recruiting-store-sales-forecasting \
  -p data

import zipfile
data_dir = "data"
while any(f.endswith(".zip") for f in os.listdir(data_dir)):
    for fname in list(os.listdir(data_dir)):
        if fname.endswith(".zip"):
            fpath = os.path.join(data_dir, fname)
            with zipfile.ZipFile(fpath, "r") as z:
                z.extractall(data_dir)
            os.remove(fpath)

%pip install -q dagshub
!pip install mlflow --quiet

import dagshub
import mlflow

dagshub.init(repo_owner="amama22", repo_name="walmart-forecasting", mlflow=True)
mlflow.set_experiment("NBEATS_Training")

Cloning into 'walmart-forecasting'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 36 (delta 13), reused 19 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 215.58 KiB | 1.26 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/walmart-forecasting
100% 2.70M/2.70M [00:00<00:00, 223MB/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d8bcccda-ca9d-44f1-9bd6-9df3bfff4e9b&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5a7d671e37fb20a6af76fc88a5046fc44e6b3c535e65da3ba1761ed565fde631




Accessing as amama22

Initialized MLflow to track repo "amama22/walmart-forecasting"

Repository amama22/walmart-forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/c57d5a176c2e4b8aa10b3c90deaccd59', creation_time=1783801585600, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783801585600, lifecycle_stage='active', name='NBEATS_Training', tags={}, trace_location=None, workspace='default'>

In [2]:
%pip install -q "u8darts[torch]"

import sys
sys.path.append(".")
import numpy as np
import pandas as pd
from darts import TimeSeries
from darts.models import NBEATSModel

from src.data_prep import load_raw_data, merge_all
from src.evaluation import wmae

train, test, features, stores = load_raw_data(data_dir="data")
train_merged = merge_all(train, features, stores)

df = train_merged[["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]].copy()
df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
df = df.sort_values(["unique_id", "Date"])

HORIZON = 38

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.5/744.5 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.6/413.6 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 33.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour i

In [3]:
full_date_range = pd.date_range(df["Date"].min(), df["Date"].max(), freq="W-FRI")

def reindex_global(group):
    group = group.set_index("Date").reindex(full_date_range)
    group["Weekly_Sales"] = group["Weekly_Sales"].fillna(0)
    group["IsHoliday"] = group["IsHoliday"].astype("boolean").fillna(False).astype(bool)
    group.index.name = "Date"
    return group

df_global = (
    df.groupby("unique_id", group_keys=True)
    .apply(reindex_global, include_groups=False)
    .reset_index()
)

series_list_raw = TimeSeries.from_group_dataframe(
    df_global, group_cols="unique_id", time_col="Date", value_cols="Weekly_Sales", freq="W-FRI",
)
unique_ids = df_global["unique_id"].unique()
print("Series:", len(series_list_raw), "| length each:", len(series_list_raw[0]))

Series: 3331 | length each: 143


In [4]:
last_date = df_global["Date"].max()
val_end = last_date
val_start = val_end - pd.Timedelta(days=HORIZON * 7)
train_cutoff = val_start

train_series_list = [s.drop_after(pd.Timestamp(train_cutoff)) for s in series_list_raw]
print("Train cutoff:", train_cutoff.date(), "| min train length:", min(len(s) for s in train_series_list))

Train cutoff: 2012-02-03 | min train length: 104


In [5]:
best_params = {
    "input_chunk_length": 46,
    "num_stacks": 2,
    "num_blocks": 2,
    "layer_widths": 128,
    "lr": 0.0009288157002342496,
}
best_wmae = 1767.9875285428097

print("Using trial 12 as final HPO result (best of 26 completed trials, runtime disconnected before trial 27):")
print(best_params)
print("Best WMAE:", best_wmae)

Using trial 12 as final HPO result (best of 26 completed trials, runtime disconnected before trial 27):
{'input_chunk_length': 46, 'num_stacks': 2, 'num_blocks': 2, 'layer_widths': 128, 'lr': 0.0009288157002342496}
Best WMAE: 1767.9875285428097


In [6]:
with mlflow.start_run(run_name="NBEATS_Final_Model"):
    mlflow.log_params(best_params)
    mlflow.log_param("n_epochs", 100)
    mlflow.log_param("output_chunk_length", HORIZON)
    mlflow.log_param("train_cutoff", str(train_cutoff.date()))

    final_model = NBEATSModel(
        input_chunk_length=best_params["input_chunk_length"],
        output_chunk_length=HORIZON,
        num_stacks=best_params["num_stacks"],
        num_blocks=best_params["num_blocks"],
        layer_widths=best_params["layer_widths"],
        optimizer_kwargs={"lr": best_params["lr"]},
        n_epochs=100,
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
    )
    final_model.fit(train_series_list)

    context_series = [s.drop_after(pd.Timestamp(val_start)) for s in series_list_raw]
    preds = final_model.predict(n=HORIZON, series=context_series)

    y_true, y_pred, is_holiday = [], [], []
    for uid, raw_series, pred in zip(unique_ids, series_list_raw, preds):
        actual_slice = raw_series.slice(pd.Timestamp(val_start) + pd.Timedelta(days=7), pd.Timestamp(val_end))
        common_len = min(len(actual_slice), len(pred))
        y_true.extend(actual_slice.values().flatten()[:common_len])
        y_pred.extend(pred.values().flatten()[:common_len])
        holiday_vals = df_global[(df_global["unique_id"] == uid) & (df_global["Date"] > val_start) & (df_global["Date"] <= val_end)]["IsHoliday"].values
        is_holiday.extend(holiday_vals[:common_len])

    final_wmae = wmae(y_true, y_pred, is_holiday)
    mlflow.log_metric("final_wmae", final_wmae)
    print(f"Final N-BEATS WMAE (100 epochs, held-out fold): {final_wmae:.2f}")

    final_model.save("nbeats_final_model.pt")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 229 K  | train
-------------------------------------------------------------
228 K     Trainable params
921       Non-trainable params
229 K     Total params
0.918     Total estimated m

Final N-BEATS WMAE (100 epochs, held-out fold): 1878.02
🏃 View run NBEATS_Final_Model at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/28da01bd2f16409b9e44b79850cfff51
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


In [7]:
with mlflow.start_run(run_name="NBEATS_Final_Model_15epochs"):
    mlflow.log_params(best_params)
    mlflow.log_param("n_epochs", 15)
    mlflow.log_param("output_chunk_length", HORIZON)
    mlflow.log_param("train_cutoff", str(train_cutoff.date()))
    mlflow.log_param("note", "100-epoch refit scored worse (1878.02) than 15-epoch HPO trial (1767.99) -- reverted to 15 epochs")

    final_model = NBEATSModel(
        input_chunk_length=best_params["input_chunk_length"],
        output_chunk_length=HORIZON,
        num_stacks=best_params["num_stacks"],
        num_blocks=best_params["num_blocks"],
        layer_widths=best_params["layer_widths"],
        optimizer_kwargs={"lr": best_params["lr"]},
        n_epochs=15,
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
    )
    final_model.fit(train_series_list)

    context_series = [s.drop_after(pd.Timestamp(val_start)) for s in series_list_raw]
    preds = final_model.predict(n=HORIZON, series=context_series)

    y_true, y_pred, is_holiday = [], [], []
    for uid, raw_series, pred in zip(unique_ids, series_list_raw, preds):
        actual_slice = raw_series.slice(pd.Timestamp(val_start) + pd.Timedelta(days=7), pd.Timestamp(val_end))
        common_len = min(len(actual_slice), len(pred))
        y_true.extend(actual_slice.values().flatten()[:common_len])
        y_pred.extend(pred.values().flatten()[:common_len])
        holiday_vals = df_global[(df_global["unique_id"] == uid) & (df_global["Date"] > val_start) & (df_global["Date"] <= val_end)]["IsHoliday"].values
        is_holiday.extend(holiday_vals[:common_len])

    final_wmae = wmae(y_true, y_pred, is_holiday)
    mlflow.log_metric("final_wmae", final_wmae)
    print(f"Final N-BEATS WMAE (15 epochs): {final_wmae:.2f}")

    final_model.save("nbeats_final_model.pt")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 229 K  | train
-------------------------------------------------------------
228 K     Trainable params
921       Non-trainable params
229 K     Total params
0.918     Total estimated m

Final N-BEATS WMAE (15 epochs): 1767.99
🏃 View run NBEATS_Final_Model_15epochs at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/f157624e1c3f4246b4c506a800b6c87c
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


In [8]:
import mlflow.pyfunc

class NBEATSWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from darts.models import NBEATSModel
        self.model = NBEATSModel.load(context.artifacts["model_path"])

    def predict(self, context, model_input):

        from darts import TimeSeries
        import pandas as pd

        series_list = TimeSeries.from_group_dataframe(
            model_input, group_cols="unique_id", time_col="Date", value_cols="Weekly_Sales", freq="W-FRI",
        )
        preds = self.model.predict(n=38, series=series_list)

        unique_ids = model_input["unique_id"].unique()
        results = []
        for uid, pred in zip(unique_ids, preds):
            for date, value in zip(pred.time_index, pred.values().flatten()):
                results.append({"unique_id": uid, "Date": date, "Weekly_Sales_Pred": value})

        return pd.DataFrame(results)


with mlflow.start_run(run_name="NBEATS_Registration"):
    mlflow.log_params(best_params)
    mlflow.log_param("n_epochs", 15)
    mlflow.log_metric("held_out_wmae", final_wmae)

    mlflow.pyfunc.log_model(
        artifact_path="nbeats_pipeline",
        python_model=NBEATSWrapper(),
        artifacts={"model_path": "nbeats_final_model.pt"},
        registered_model_name="walmart-nbeats-store-sales",
    )

    print("Logged and registered as 'walmart-nbeats-store-sales'")
    print(f"Held-out fold WMAE: {final_wmae:.2f}")

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/07/12 06:02:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 06:02:42 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/07/12 06:03:02 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'walmart-nbeats-store-sales'.
2026/07/12 06:03:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart-nbeats-store-sales, version 1
Created version '1' of model 'walmart-nbeats-store-sales'.


Logged and registered as 'walmart-nbeats-store-sales'
Held-out fold WMAE: 1767.99
🏃 View run NBEATS_Registration at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/800e1d38024941dbbc82c37fc02be733
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2


In [9]:
n_test_weeks = test["Date"].nunique()
print("Actual test set weeks:", n_test_weeks)
print("Test date range:", test["Date"].min(), "to", test["Date"].max())

Actual test set weeks: 39
Test date range: 2012-11-02 00:00:00 to 2013-07-26 00:00:00


In [10]:
with mlflow.start_run(run_name="NBEATS_Submission_Refit"):
    mlflow.log_params(best_params)
    mlflow.log_param("n_epochs", 15)
    mlflow.log_param("note", "refit on full train history for Kaggle submission")

    submission_model = NBEATSModel(
        input_chunk_length=best_params["input_chunk_length"],
        output_chunk_length=HORIZON,
        num_stacks=best_params["num_stacks"],
        num_blocks=best_params["num_blocks"],
        layer_widths=best_params["layer_widths"],
        optimizer_kwargs={"lr": best_params["lr"]},
        n_epochs=15,
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "devices": 1, "enable_progress_bar": False},
    )
    submission_model.fit(series_list_raw)  # full history, no truncation this time

test_preds = submission_model.predict(n=n_test_weeks, series=series_list_raw)

rows = []
for uid, pred in zip(unique_ids, test_preds):
    store, dept = uid.split("_")
    for date, value in zip(pred.time_index, pred.values().flatten()):
        rows.append({
            "Id": f"{store}_{dept}_{date.strftime('%Y-%m-%d')}",
            "Weekly_Sales": max(value, 0),  # clip negatives, same rule as the tree models
        })

nbeats_submission = pd.DataFrame(rows)
print("Submission shape:", nbeats_submission.shape)
print("Expected shape:", test.shape[0])
nbeats_submission.to_csv("submission_nbeats.csv", index=False)
nbeats_submission.head()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 229 K  | train
-------------------------------------------------------------
228 K     Trainable params
921       Non-trainable params
229 K     Total params
0.918     Total estimated m

🏃 View run NBEATS_Submission_Refit at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2/runs/d43c2c89d76d4e4f9c5731a6f641915b
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/2
Submission shape: (129909, 2)
Expected shape: 115064


,Id,Weekly_Sales
0,10_1_2012-11-02,36833.016574
1,10_1_2012-11-09,27823.879286
2,10_1_2012-11-16,58344.486848
3,10_1_2012-11-23,74575.937622
4,10_1_2012-11-30,83055.287583


In [11]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission_nbeats.csv -m "N-BEATS, darts, tuned, WMAE~1768 single-fold"

100% 4.32M/4.32M [00:00<00:00, 5.78MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting

In [12]:
train_ids = set(df_global["unique_id"].unique())
test["unique_id"] = test["Store"].astype(str) + "_" + test["Dept"].astype(str)
test_ids = set(test["unique_id"].unique())

print("Train series:", len(train_ids))
print("Test series:", len(test_ids))
print("In test but not train (can't be forecasted by our model):", len(test_ids - train_ids))
print("In train but not test (predicted but not needed):", len(train_ids - test_ids))

Train series: 3331
Test series: 3169
In test but not train (can't be forecasted by our model): 11
In train but not test (predicted but not needed): 173


In [13]:
pred_lookup = {}
for uid, pred in zip(unique_ids, test_preds):
    store, dept = uid.split("_")
    for date, value in zip(pred.time_index, pred.values().flatten()):
        pred_lookup[(store, dept, date.strftime("%Y-%m-%d"))] = max(value, 0)

rows = []
missing_count = 0
for _, row in test.iterrows():
    key = (str(row["Store"]), str(row["Dept"]), row["Date"].strftime("%Y-%m-%d"))
    pred_value = pred_lookup.get(key)
    if pred_value is None:
        pred_value = 0.0  # series never seen in training -- model has no basis to forecast it
        missing_count += 1
    rows.append({"Id": f"{key[0]}_{key[1]}_{key[2]}", "Weekly_Sales": pred_value})

nbeats_submission = pd.DataFrame(rows)
print("Submission shape:", nbeats_submission.shape, "| expected:", test.shape[0])
print("Rows filled with fallback 0 (unseen series):", missing_count)
nbeats_submission.to_csv("submission_nbeats.csv", index=False)
nbeats_submission.head()

Submission shape: (115064, 2) | expected: 115064
Rows filled with fallback 0 (unseen series): 36


,Id,Weekly_Sales
0,1_1_2012-11-02,25281.411595
1,1_1_2012-11-09,26091.713576
2,1_1_2012-11-16,30338.380080
3,1_1_2012-11-23,34057.542239
4,1_1_2012-11-30,36597.722647


In [14]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission_nbeats.csv -m "N-BEATS, darts, tuned, WMAE~1768 single-fold"

100% 3.83M/3.83M [00:00<00:00, 5.06MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting